In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from google import genai
client = genai.Client()

In [ ]:
def llm(prompt):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    return response.text

In [4]:
output = llm('Tell me a joke!')
print(output)

Why don't scientists trust atoms?

Because they make up everything!


In [5]:
context = '''
    I just discovered the course. Can I still join?
    Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

    Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
    You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

    What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
    The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

    Cloud alternatives with GPU
    Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
'''

In [6]:
output = llm("I just discovered this databricks course, can I join it now ?")
print(output)

It depends on the specific Databricks course you've discovered!

Databricks offers courses through various platforms and formats, and whether you can join "now" depends on which type it is:

1.  **Databricks Academy (Self-Paced Courses):** Many official Databricks courses are hosted on [Databricks Academy](https://www.databricks.com/resources/training) and are **self-paced**. For these, you can usually enroll and start anytime you like, as long as you have a Databricks account (which is free to create for the Academy).

2.  **Coursera, edX, or other MOOC Platforms:** If the course is on a platform like Coursera, most are also **self-paced** and allow continuous enrollment. You can typically join, audit, or pay for a certificate whenever you're ready.

3.  **Instructor-Led Training/Workshops:** Some Databricks training sessions are live, instructor-led workshops or multi-day courses. These will have **fixed start and end dates** and a specific registration window. If it's this type, you

In [7]:
question = "I just discovered the course, can I join now ?"

prompt = f"""
    Your task is to answer questions from the course participants
    based on the provided context.

    Use the context to find relevant information and provide accurate
    answers. If the answer is not found in the context,
    respond with "I don't know."

    Question:
    {question}

    Context:
    {context}
"""

In [8]:
answer = llm(prompt)
print(answer)

Yes, you can join now. If you want to receive a certificate, you need to submit your project while submissions are still being accepted. You can also just start learning and submitting homework without formal registration.


In [9]:
# def rag(question):
#     search_results = search(question)
#     user_prompt = build_prompt(question, search_results)
#     # return llm(prompt)


In [10]:
import requests
docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [11]:
courses_raw

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 404},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 103},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

In [12]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()
    
    documents.extend(course_data)

len(documents)

1368

In [13]:
documents[1]

{'id': 'bfafa427b3',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: What are the prerequisites for this course?',
 'answer': "To get the most out of this course, you should have:\n\n- Basic coding experience\n- Familiarity with SQL\n- Experience with Python (helpful but not required)\n\nNo prior data engineering experience is necessary. See [Readme on GitHub](https://github.com/DataTalksClub/data-engineering-zoomcamp/blob/main/README.md#prerequisites).\n\nIf you have these basics, you're ready to start — you don't need to master everything up front. The course covers Git and GitHub (see *How do I use Git/GitHub for this course?*), and you'll pick up the command-line/Linux basics you need during the setup modules."}

In [14]:
from minsearch import Index

index = Index(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course']
)

index.fit(documents)

In [15]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict, 
        num_results=5
    )

In [ ]:
search_result = search(question)

[{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}, {'id': '977bf7786c', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?', 'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."}, {'id': '69d122f12e', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?', 'answer': 'No, you can only get a certificate 

In [17]:
INSTRUCTIONS = """
    Your task is to answer questions from the course participants
    based on the provided context.

    Use the context to find relevant information and provide accurate
    answers. If the answer is not found in the context,
    respond with "I don't know."
"""

In [18]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [19]:
def build_prompt(question_local, search_result):
    context_local = build_context(search_result)
    USER_PROMPT_TEMPLATE = f"""
        Question:
        {question_local}

        Context:
        {context_local}
    """
    
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context_local=context_local
    )
    return prompt.strip()

In [20]:
prompt_pipeline = build_prompt(question, search_result)
print(prompt_pipeline)

Question:
        I just discovered the course, can I join now ?

        Context:
        General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) afte

In [40]:
from google.genai import types

def llm_new(instructions, user_prompt, model):
    # message_history = [
    #     {"role": "developer", "content": instructions},
    #     {"role": "user", "content": user_prompt}
    # ]
    response = client.models.generate_content(
        model=model,
        contents=user_prompt,
        config=types.GenerateContentConfig(
            system_instruction=instructions
        )
    )
    
    return response.text

In [41]:
def rag(query, model="gemini-2.5-flash"):
    search_results = search(query)
    prompt_local = build_prompt(query, search_results)
    answer = llm_new(INSTRUCTIONS, prompt_local, model)
    return answer

In [42]:
rag("How do I get a certificate ?")

'You can only get a certificate if you finish the course with a "live" cohort and pass the Capstone project. Certificates are not awarded for the self-paced mode.'